In [0]:
create schema if not exists 03_gold_catalog.analytics

In [0]:
select count(*) from `03_gold_catalog`.facts.fact_opportunity

In [0]:
with cte1 as(
select end_date,row_number() over(order by end_date) as rn from `03_gold_catalog`.facts.fact_opportunity
)
select * from cte1 where rn=1 or rn=150000



In [0]:
with cte1 as(
  select customer_sk,min(start_date) as first_start
  from `03_gold_catalog`.facts.fact_opportunity
  group by customer_sk
)
select count(*) as customers_gained
from cte1
where first_start between "2023-01-01" and "2023-03-01"

In [0]:
with ended_customers as(
  select distinct customer_sk
  from `03_gold_catalog`.facts.fact_opportunity
  where end_date between "2023-01-01" and " 2023-03-01"
),
active_customers as(
  select distinct customer_sk
  from `03_gold_catalog`.facts.fact_opportunity
  where start_date > "2023-03-01"
)
select count(*) as lost_customers
from ended_customers e 
left join active_customers a on e.customer_sk = a.customer_sk
where a.customer_sk is null


In [0]:
select customer_sk,count(*) from `03_gold_catalog`.dimensions.dim_customer
group by customer_sk

In [0]:
with repetive_customer as(
  select customer_sk  , count(distinct opportunity_id ) as no_of_times_repeated , sum(revenue_amount) as tot_revenue
  from `03_gold_catalog`.facts.fact_opportunity
  group by customer_sk
)
select d.customer_id,d.customer_name ,r.tot_revenue, r.customer_sk
from `03_gold_catalog`.dimensions.dim_customer as d
join repetive_customer as r on d.customer_sk = r.customer_sk
where no_of_times_repeated > 1
order by tot_revenue desc
limit 10

In [0]:
with customer_revenue as (
  select 
    customer_sk,
    sum(
      revenue_amount / 
      (datediff(end_date, start_date) / 30.0)
    ) as mrr
  from 03_gold_catalog.facts.fact_opportunity
  where end_date > start_date
  group by customer_sk
)

select 
  d.customer_id,
  d.customer_name,
  r.mrr
from 03_gold_catalog.dimensions.dim_customer d
join customer_revenue r
  on d.customer_sk = r.customer_sk
order by r.mrr desc
limit 10;

In [0]:
with cte1 as(
select customer_sk,year(start_date)as year,count(*) as no_of_renewals
from 03_gold_catalog.facts.fact_opportunity
group by customer_sk,year
order by customer_sk,year
),
cte2 as(
  select * , lag(no_of_renewals) over(partition by customer_sk order by year) as prev_year_renewals
  from cte1
),
cte3 as(
  select *,
    case 
      when prev_year_renewals is null then null
      when no_of_renewals > prev_year_renewals then 1
      when no_of_renewals < prev_year_renewals then -1
      else 0
    end as trend
  from cte2
),
cte4 as (
  select customer_sk ,min(trend) as min_trend ,max(trend) as max_trend
  from cte3
  where trend is not null
  group by customer_sk
),
cte5 as(
select d.customer_id,
  case 
    when min_trend = 1 and max_trend = 1 then 'up'
    when min_trend = -1 and max_trend = -1 then 'down'
    else 'stable'
  end as growth_trend
from cte4 c
join 03_gold_catalog.dimensions.dim_customer d on c.customer_sk = d.customer_sk
order by customer_id
)
select * from cte5 where growth_trend="up"

In [0]:
-- recurring revenue monthly
create or replace table `03_gold_catalog`.support_tables.mrr as
with cte1 as(
  select
    customer_sk,
    date_trunc('month',start_date) as start_month,
    case 
      when contract_term = 'Yearly' then 12
      else 1
    end as no_of_months,
    case 
      when contract_term = 'Yearly' then revenue_amount/12
      else revenue_amount
    end as monthly_revenue
  from `03_gold_catalog`.facts.fact_opportunity
  where close_status='Won'
  order by customer_sk,start_date
),
cte2 as(
  select customer_sk,
  monthly_revenue,
  explode(sequence(start_month,add_months(start_month, no_of_months - 1), interval 1 month)) as report_month
  from cte1
)
select 
  year(report_month) as year,
  month(report_month) as month,
  sum(monthly_revenue) as total_monthly_revenue
from cte2
group by year,month
order by year,month

In [0]:
-- recurring revenue yearly
create or replace table `03_gold_catalog`.support_tables.yrr as
with cte1 as(
  select
    customer_sk,
    date_trunc('month',start_date) as start_month,
    case 
      when contract_term = 'Yearly' then 12
      else 1
    end as no_of_months,
    case 
      when contract_term = 'Yearly' then revenue_amount/12
      else revenue_amount
    end as monthly_revenue
  from `03_gold_catalog`.facts.fact_opportunity
  where close_status='Won'
  order by customer_sk,start_date
),
cte2 as(
  select customer_sk,
  monthly_revenue,
  explode(sequence(start_month,add_months(start_month, no_of_months - 1), interval 1 month)) as report_month
  from cte1
)
select 
  case 
    when month(report_month) >= 4 then year(report_month)
    else year(report_month)-1
  end as financial_year,
  sum(monthly_revenue) as total_yearly_revenue
from cte2
group by financial_year

In [0]:
-- 1
with cte1 as(
  select
    customer_sk,
    start_date,
    end_date,
    lag (start_date) over (partition by customer_sk order by start_date) as prev_start_date,
    lead (end_date) over (partition by customer_sk order by end_date) as next_end_date
  from `03_gold_catalog`.facts.fact_opportunity 
  where close_status='Won'
),
cte2 as (
  select
    count(distinct customer_sk) as customers_gained
  from cte1
  where year(start_date) = 2025 and prev_start_date is null
),
cte3 as(
  select 
    count(distinct customer_sk) as customers_lost
  from cte1
  where year(end_date) = 2025 and next_end_date is null
)
select 
  customers_gained,
  customers_lost
from cte2 cross join cte3

In [0]:
select count(distinct customer_sk) from `03_gold_catalog`.facts.fact_opportunity where close_status='Won'

In [0]:
-- 2
with cte1 as(
  select
    customer_sk,
    year(end_date) as year,
    lead(end_date) over (partition by customer_sk order by end_date) as next_end_date
  from `03_gold_catalog`.facts.fact_opportunity
  where close_status='Won'
  order by customer_sk,start_date
),
cte2 as (
  select *
  from cte1
  where next_end_date is null
)
select 
  year,
  count(distinct customer_sk) as customers_lost
from cte2
group by year
order by year;

In [0]:
-- 3
